# 04. SQL Analysis & Data Visualisation

**Objective:** Combine SQL-based analysis with Python visualisation to examine distributions, relationships, composition and comparisons in the survey data.

## 1. Connect to the SQLite survey database

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
database_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/LargeData/m4_survey_data.sqlite"
local_db = "m4_survey_data.sqlite"

> The original course workflow downloads the SQLite database. In a standard local Python environment, download the database from the source URL and set `local_db` to its local path.

In [ ]:
# Example for a local copy of the database:
# conn = sqlite3.connect(local_db)

# If the database is already available locally:
conn = sqlite3.connect(local_db)

## 2. SQL: count records

In [ ]:
query = """
SELECT COUNT(*) AS respondent_count
FROM master
"""
pd.read_sql_query(query, conn)

## 3. SQL: inspect tables

In [ ]:
query = """
SELECT name AS table_name
FROM sqlite_master
WHERE type = "table"
"""
pd.read_sql_query(query, conn)

## 4. SQL: group respondents by age

In [ ]:
query = """
SELECT Age, COUNT(*) AS respondent_count
FROM master
GROUP BY Age
ORDER BY Age
"""
age_summary = pd.read_sql_query(query, conn)
age_summary.head()

## 5. SQL: inspect table definition

In [ ]:
query = """
SELECT sql
FROM sqlite_master
WHERE name = "master"
"""
table_definition = pd.read_sql_query(query, conn)
print(table_definition.iat[0, 0])

## 6. Load survey data for visualisation

In [ ]:
survey_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/LargeData/m2_survey_data.csv"
df = pd.read_csv(survey_url)
df.head()

## 7. Distribution: converted compensation

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df["ConvertedComp"].dropna())
plt.xlabel("Converted compensation")
plt.ylabel("Frequency")
plt.title("Distribution of Converted Compensation")
plt.tight_layout()
plt.show()

## 8. Distribution: age

In [ ]:
plt.figure(figsize=(10, 6))
df["Age"].plot(kind="box")
plt.ylabel("Age")
plt.title("Age Distribution")
plt.tight_layout()
plt.show()

## 9. Relationship: age and work-week hours

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(x="Age", y="WorkWeekHrs", data=df, scatter_kws={"alpha": 0.35})
plt.xlabel("Age")
plt.ylabel("Work-week hours")
plt.title("Age vs Work-week Hours")
plt.tight_layout()
plt.show()

## 10. Bubble plot: work hours and code review hours

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(x="WorkWeekHrs", y="CodeRevHrs", size="Age", data=df, sizes=(20, 300), alpha=0.5)
plt.xlabel("Work-week hours")
plt.ylabel("Code review hours")
plt.title("Work-week Hours vs Code Review Hours")
plt.tight_layout()
plt.show()

## 11. Composition: databases desired next year

In [ ]:
database_counts = df["DatabaseDesireNextYear"].value_counts().head(5)
plt.figure(figsize=(9, 7))
plt.pie(database_counts, labels=database_counts.index, autopct="%1.1f%%", startangle=140)
plt.title("Top 5 Databases Desired Next Year")
plt.tight_layout()
plt.show()

## 12. Targeted checks

In [ ]:
sql_count = df["LanguageWorkedWith"].fillna("").str.contains("SQL", case=False).sum()
print(f"Respondents whose worked-with languages include SQL: {sql_count:,}")

In [ ]:
mysql_count = df["DatabaseWorkedWith"].fillna("").eq("MySQL").sum()
print(f"Respondents whose database field is MySQL: {mysql_count:,}")

## 13. Comparison: median compensation for ages 45–60

In [ ]:
age_filtered = df[df["Age"].between(45, 60)]
median_comp_by_age = age_filtered.groupby("Age")["ConvertedComp"].median()
plt.figure(figsize=(10, 6))
median_comp_by_age.plot(kind="bar")
plt.xlabel("Age")
plt.ylabel("Median converted compensation")
plt.title("Median Compensation by Age: 45–60")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 14. Comparison: respondent main branch

In [ ]:
main_branch = df["MainBranch"].value_counts()
plt.figure(figsize=(10, 6))
main_branch.plot(kind="barh")
plt.xlabel("Number of respondents")
plt.ylabel("Main branch")
plt.title("Respondents by Main Branch")
plt.tight_layout()
plt.show()

In [ ]:
conn.close()

## Key takeaway

This notebook demonstrates how SQL can be used to interrogate a relational dataset and how Python visualisations can then communicate the resulting patterns through distributions, relationships, composition and comparison charts.